# Experiment 6 (Dept. Handout) — Dimensionality Reduction and Model Evaluation (With and Without PCA)

Generic `assn7_experiment()` function — works on any tabular classification dataset. Dataset used here: the **Wisconsin Diagnostic Breast Cancer Dataset** (569 samples, 30 features, loaded directly from `sklearn.datasets`).

Trains and tunes 10 classifiers (SVM, Naive Bayes, KNN, Logistic Regression, Decision Tree, Random Forest, AdaBoost, Gradient Boosting, XGBoost, Stacking) under both the original 30-feature space (**No-PCA**) and a PCA-reduced feature space (**With-PCA**, components chosen to explain 95% variance), each with 5-fold cross-validation, and compares whether PCA helped or hurt each model.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
)
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

# xgboost is not part of scikit-learn -- install it once if missing:
#   pip install xgboost

## `assn7_experiment()`

STEP 1: Preprocess (split + scale)
STEP 2: PCA — choose components for 95% variance, scree plot
STEP 3: Define a small hyperparameter grid per model
STEP 4: Tune + 5-fold CV every model under No-PCA AND With-PCA
STEP 5: Stacking (fixed structure, evaluated the same way)
STEP 6: Build the summary table (fold-wise + averages, No-PCA vs With-PCA)
STEP 7: Confusion matrices + ROC curves for selected models
STEP 8: No-PCA vs With-PCA accuracy comparison chart

In [ ]:
def assn7_experiment(
    df,
    target_column,
    variance_target=0.95,
    test_size=0.20,
    cv_folds=5,
    random_state=42,
    roc_cm_models=("SVM", "Stacking")
):
    """
    Experiment 6 (dept. handout) -- Dimensionality Reduction and Model
    Evaluation, With and Without PCA. Works on ANY tabular classification
    dataset (df + target_column).

    STEP 1: Preprocess (split + scale)
    STEP 2: PCA -- choose components for `variance_target` variance, scree plot
    STEP 3: Define a small hyperparameter grid per model
    STEP 4: Tune + 5-fold CV every model under No-PCA AND With-PCA
    STEP 5: Stacking (fixed structure, evaluated the same way)
    STEP 6: Build the summary table (fold-wise + averages, No-PCA vs With-PCA)
    STEP 7: Confusion matrices + ROC curves for selected models
    STEP 8: No-PCA vs With-PCA accuracy comparison chart
    """

    # ==============================================================
    # STEP 1: PREPROCESS
    # ==============================================================
    X = df.drop(columns=[target_column])
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    print("Train shape:", X_train_s.shape, "Test shape:", X_test_s.shape)

    # ==============================================================
    # STEP 2: PCA
    # ==============================================================
    # Fit PCA with ALL components first just to see how much variance each
    # one explains, then pick the smallest number of components whose
    # CUMULATIVE explained variance crosses our target (e.g. 95%). This is
    # a common, defensible way to choose "how many components" without
    # guessing a fixed number.
    pca_full = PCA().fit(X_train_s)
    cum_var = np.cumsum(pca_full.explained_variance_ratio_)
    n_components = int(np.argmax(cum_var >= variance_target) + 1)
    print(f"PCA components for {variance_target*100:.0f}% variance:", n_components,
          f"(explains {cum_var[n_components-1]*100:.2f}%)")

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(cum_var) + 1), cum_var, "o-")
    plt.axhline(variance_target, color="r", linestyle="--", label=f"{variance_target*100:.0f}% variance target")
    plt.axvline(n_components, color="g", linestyle="--", label=f"n_components={n_components}")
    plt.xlabel("Number of Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("PCA Scree Plot")
    plt.legend()
    plt.show()

    # Refit PCA on the train set only (never on test -> avoids data leakage),
    # then just transform() the test set using what PCA learned from train.
    pca = PCA(n_components=n_components, random_state=random_state)
    X_train_pca = pca.fit_transform(X_train_s)
    X_test_pca = pca.transform(X_test_s)
    print("With-PCA train shape:", X_train_pca.shape)

    # ==============================================================
    # STEP 3: HYPERPARAMETER GRIDS (kept small so 10 models x 2 settings
    # x 5-fold CV finishes in a reasonable time -- widen these if you want
    # a more thorough search)
    # ==============================================================
    model_defs = {
        "SVM": (SVC(probability=True, random_state=random_state),
                {"kernel": ["rbf"], "C": [1, 10], "gamma": ["scale", "auto"]}),
        "Naive Bayes": (GaussianNB(),
                        {"var_smoothing": [1e-9, 1e-8, 1e-7]}),
        "KNN": (KNeighborsClassifier(),
                {"n_neighbors": [3, 5, 7], "weights": ["uniform", "distance"]}),
        "Logistic Regression": (LogisticRegression(max_iter=5000, random_state=random_state),
                                 {"C": [0.1, 1, 10]}),
        "Decision Tree": (DecisionTreeClassifier(random_state=random_state),
                           {"max_depth": [3, 5, None], "min_samples_split": [2, 5]}),
        "Random Forest": (RandomForestClassifier(random_state=random_state),
                           {"n_estimators": [50, 100], "max_depth": [5, None]}),
        "AdaBoost": (AdaBoostClassifier(random_state=random_state),
                     {"n_estimators": [50, 100], "learning_rate": [0.1, 1.0]}),
        "Gradient Boosting": (GradientBoostingClassifier(random_state=random_state),
                               {"n_estimators": [50, 100], "learning_rate": [0.1, 1.0], "max_depth": [2, 3]}),
        "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=random_state),
                    {"n_estimators": [50, 100], "learning_rate": [0.1, 1.0], "max_depth": [3, 5]}),
    }

    # ==============================================================
    # STEP 4: TUNE + 5-FOLD CV, for every model, under BOTH settings
    # ==============================================================
    def tune_and_cv(base_model, grid, Xtr, ytr, Xte, yte):
        # GridSearchCV finds the best hyperparameters using its own internal
        # cross-validation.
        start = time.time()
        gs = GridSearchCV(base_model, grid, cv=cv_folds, scoring="accuracy", n_jobs=-1)
        gs.fit(Xtr, ytr)
        tune_time = time.time() - start
        best = gs.best_estimator_

        # Then, separately, we run our OWN 5-fold CV on the best model so we
        # have the individual fold scores to report (GridSearchCV only keeps
        # the average internally, not each fold's score for the winner).
        fold_scores = cross_val_score(
            best, Xtr, ytr,
            cv=StratifiedKFold(cv_folds, shuffle=True, random_state=random_state),
            scoring="accuracy"
        )

        test_pred = best.predict(Xte)
        return {
            "best_params": gs.best_params_,
            "fold_scores": fold_scores,
            "avg_cv_accuracy": fold_scores.mean(),
            "test_accuracy": accuracy_score(yte, test_pred),
            "test_f1": f1_score(yte, test_pred),
            "tune_time": tune_time,
            "model": best
        }

    results_no_pca = {}
    results_with_pca = {}

    for name, (model, grid) in model_defs.items():
        print(f"\n=== {name} ===")
        r1 = tune_and_cv(model, grid, X_train_s, y_train, X_test_s, y_test)
        print("No-PCA   best params:", r1["best_params"], " avg CV acc:", round(r1["avg_cv_accuracy"], 4))
        results_no_pca[name] = r1

        r2 = tune_and_cv(model, grid, X_train_pca, y_train, X_test_pca, y_test)
        print("With-PCA best params:", r2["best_params"], " avg CV acc:", round(r2["avg_cv_accuracy"], 4))
        results_with_pca[name] = r2

    # ==============================================================
    # STEP 5: STACKING (fixed base learners + meta-learner, no grid search)
    # ==============================================================
    def build_stack():
        return StackingClassifier(
            estimators=[
                ("svm", SVC(probability=True, random_state=random_state)),
                ("nb", GaussianNB()),
                ("dt", DecisionTreeClassifier(random_state=random_state))
            ],
            final_estimator=LogisticRegression(max_iter=5000, random_state=random_state),
            cv=cv_folds
        )

    def eval_stack(Xtr, ytr, Xte, yte):
        stack = build_stack()
        start = time.time()
        stack.fit(Xtr, ytr)
        fit_time = time.time() - start
        fold_scores = cross_val_score(
            build_stack(), Xtr, ytr,
            cv=StratifiedKFold(cv_folds, shuffle=True, random_state=random_state),
            scoring="accuracy"
        )
        pred = stack.predict(Xte)
        return {
            "best_params": {}, "fold_scores": fold_scores,
            "avg_cv_accuracy": fold_scores.mean(),
            "test_accuracy": accuracy_score(yte, pred),
            "test_f1": f1_score(yte, pred),
            "tune_time": fit_time, "model": stack
        }

    print("\n=== Stacking ===")
    results_no_pca["Stacking"] = eval_stack(X_train_s, y_train, X_test_s, y_test)
    results_with_pca["Stacking"] = eval_stack(X_train_pca, y_train, X_test_pca, y_test)
    print("No-PCA   Stacking avg CV acc:", round(results_no_pca["Stacking"]["avg_cv_accuracy"], 4))
    print("With-PCA Stacking avg CV acc:", round(results_with_pca["Stacking"]["avg_cv_accuracy"], 4))

    # ==============================================================
    # STEP 6: BUILD THE SUMMARY TABLE
    # ==============================================================
    all_model_names = list(model_defs.keys()) + ["Stacking"]
    rows = []
    for name in all_model_names:
        r1, r2 = results_no_pca[name], results_with_pca[name]
        row = {"Model": name}
        for i, s in enumerate(r1["fold_scores"], start=1):
            row[f"Fold{i}_NoPCA"] = s
        row["Avg_NoPCA"] = r1["avg_cv_accuracy"]
        for i, s in enumerate(r2["fold_scores"], start=1):
            row[f"Fold{i}_WithPCA"] = s
        row["Avg_WithPCA"] = r2["avg_cv_accuracy"]
        row["TestAcc_NoPCA"] = r1["test_accuracy"]
        row["TestAcc_WithPCA"] = r2["test_accuracy"]
        row["TestF1_NoPCA"] = r1["test_f1"]
        row["TestF1_WithPCA"] = r2["test_f1"]
        rows.append(row)

    summary_df = pd.DataFrame(rows)
    print("\nSummary (5-Fold CV, No-PCA vs With-PCA)")
    display(summary_df)

    best_params_df = pd.DataFrame([{
        "Model": name,
        "Best Params (No-PCA)": results_no_pca[name]["best_params"],
        "Avg CV Acc (No-PCA)": results_no_pca[name]["avg_cv_accuracy"],
        "Best Params (With-PCA)": results_with_pca[name]["best_params"],
        "Avg CV Acc (With-PCA)": results_with_pca[name]["avg_cv_accuracy"],
    } for name in model_defs.keys()])
    print("\nBest Hyperparameters per Model")
    display(best_params_df)

    # ==============================================================
    # STEP 7: CONFUSION MATRICES + ROC CURVES for selected models
    # ==============================================================
    for tag, res_dict, Xte in [("NoPCA", results_no_pca, X_test_s), ("WithPCA", results_with_pca, X_test_pca)]:
        for sel in roc_cm_models:
            model = res_dict[sel]["model"]
            pred = model.predict(Xte)
            ConfusionMatrixDisplay.from_predictions(y_test, pred)
            plt.title(f"Confusion Matrix - {sel} ({tag})")
            plt.show()

    plt.figure(figsize=(8, 7))
    for tag, res_dict, Xte in [("NoPCA", results_no_pca, X_test_s), ("WithPCA", results_with_pca, X_test_pca)]:
        for sel in roc_cm_models:
            model = res_dict[sel]["model"]
            prob = model.predict_proba(Xte)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, prob)
            auc = roc_auc_score(y_test, prob)
            plt.plot(fpr, tpr, label=f"{sel} ({tag}, AUC={auc:.4f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves - {', '.join(roc_cm_models)} (No-PCA vs With-PCA)")
    plt.legend()
    plt.show()

    # ==============================================================
    # STEP 8: NO-PCA vs WITH-PCA ACCURACY COMPARISON CHART
    # ==============================================================
    plt.figure(figsize=(12, 6))
    x = np.arange(len(summary_df))
    plt.bar(x - 0.2, summary_df["Avg_NoPCA"] * 100, width=0.4, label="No-PCA")
    plt.bar(x + 0.2, summary_df["Avg_WithPCA"] * 100, width=0.4, label="With-PCA")
    plt.xticks(x, summary_df["Model"], rotation=45, ha="right")
    plt.ylabel("Avg 5-Fold CV Accuracy (%)")
    plt.title("No-PCA vs With-PCA: Average CV Accuracy per Model")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return {
        "summary_df": summary_df,
        "best_params_df": best_params_df,
        "results_no_pca": results_no_pca,
        "results_with_pca": results_with_pca,
        "n_components": n_components,
        "explained_variance": cum_var[n_components - 1],
        "pca": pca,
        "scaler": scaler
    }

### Usage

The Wisconsin Breast Cancer dataset is loaded directly from `sklearn.datasets` (no CSV needed).

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["diagnosis"] = data.target   # 0 = malignant, 1 = benign

exp7_output = assn7_experiment(df, "diagnosis")

exp7_output["summary_df"].to_csv("Experiment7_Summary.csv", index=False)
exp7_output["best_params_df"].to_csv("Experiment7_BestParams.csv", index=False)
exp7_output["summary_df"]